# 14. Python 套件：一次分析 22 個縣市

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begin0808/RustGIS-TW/blob/main/docs/notebooks/14-python.ipynb)

這份筆記本是「[RustGIS 臺灣教學網](https://begin0808.github.io/RustGIS-TW/tutorials/14-python/)」第 14 課的練習檔。每個灰色的程式碼格子，按左邊的 ▶ 執行，或點一下格子再按 **Shift + Enter**。請**從上往下依序執行**。

## 1. 安裝

第一格會安裝兩樣東西，約需 1 分鐘：

- **中文字型**（Noto Sans CJK）：Colab 的電腦沒有中文字型，不裝的話，地圖上的中文標註會畫不出來。
- **rustgis 套件**：RustGIS 的 Python 版。

In [ ]:
!apt-get -qq update > /dev/null 2>&1 && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1
%pip install -q rustgis

## 2. 下載縣市界線

從本站下載內政部國土測繪中心的直轄市、縣市界線（和第 2 課是同一份資料）。

In [ ]:
import os, urllib.request

url = "https://begin0808.github.io/RustGIS-TW/data/files/COUNTY_MOI_1140318.zip"
if not os.path.exists("COUNTY_MOI_1140318.zip"):
    urllib.request.urlretrieve(url, "COUNTY_MOI_1140318.zip")
print(os.path.getsize("COUNTY_MOI_1140318.zip") // 1024, "KB")

## 3. 開啟圖層

`Session` 就像一個看不到畫面的 RustGIS。`add_data` 和網頁版的「新增資料」一樣，ZIP 檔不用解壓縮。

In [ ]:
import rustgis

s = rustgis.Session()
county = s.add_data("COUNTY_MOI_1140318.zip")
print(county.name, "有", county.count, "個圖徵")
print("欄位：", county.fields)

## 4. 畫出地圖

設定底圖、顏色、外框和標註，再把畫面移到臺灣本島。格子的最後一行寫 `s`，就會顯示地圖。

In [ ]:
s.set_basemap("blank")
s.run("layer.color", layer=county.name, color="#F3EEE4")
s.run("layer.outline", layer=county.name, color="#7A7A7A", width=0.8)
s.run("labels.field", layer=county.name, field="COUNTYNAME")
s.zoom_to((119.3, 21.85, 122.1, 25.35))   # 西、南、東、北的經緯度
s

## 5. 用程式重做第 5 課

選出臺北市 → 環域 10 公里 → 清除選取 → 依位置選取。工具的名稱和參數，和網頁版、模型建置器（第 13 課）用的一樣。

In [ ]:
county.select("COUNTYNAME = '臺北市'")
buf = s.run_tool("buffer", input=county, distance=10, units="Kilometers", output="臺北市_10km")
county.clear_selection()
s.run_tool("selectByLocation", input=county, select=buf)

names = [f["properties"]["COUNTYNAME"] for f in county.to_geojson(selected=True)["features"]]
print(len(names), "個縣市：", names)

In [ ]:
s.run("layer.color", layer=buf.name, color="#E4572E55")
s.run("layer.outline", layer=buf.name, color="#C0392B", width=1.2)
s.zoom_to(buf)
s

## 6. 一次分析 22 個縣市

用 `for` 迴圈，對每個縣市都做一次「環域 10 公里 → 依位置選取」，數一數各碰到幾個**其他**縣市。

縣市界線畫得非常細，直接做環域一個縣市要好幾秒。所以先用 **簡化**（simplify）把邊界的點減少：`tolerance=0.1` 是容許誤差 0.1 公里（100 公尺）。環域的距離是 10 公里，邊界偏移 100 公尺通常不會影響結果，速度卻快了好幾百倍（本例已用原始邊界驗算過，結果完全一樣）。

In [ ]:
import time
import pandas as pd

if county.selected:        # 先清除第 5 步的選取，否則只會簡化選取的 4 個縣市
    county.clear_selection()
simple = s.run_tool("simplify", input=county, tolerance=0.1, output="縣市_簡化")

start = time.time()
rows = []
for f in simple.to_geojson()["features"]:
    name = f["properties"]["COUNTYNAME"]
    simple.select(f"COUNTYNAME = '{name}'")
    b = s.run_tool("buffer", input=simple, distance=10, units="Kilometers", output="暫存環域")
    simple.clear_selection()
    s.run_tool("selectByLocation", input=simple, select=b)
    hit = [g["properties"]["COUNTYNAME"] for g in simple.to_geojson(selected=True)["features"]]
    others = [h for h in hit if h != name]
    rows.append({"縣市": name, "10公里內的其他縣市數": len(others), "有哪些": "、".join(others)})
    simple.clear_selection()
    b.remove()
print(f"花了 {time.time() - start:.1f} 秒")

df = pd.DataFrame(rows).sort_values("10公里內的其他縣市數", ascending=False, ignore_index=True)
df

## 7. 把結果畫成地圖

把每個縣市的數字寫回圖徵的屬性，加成新圖層，再用**分級設色**（第 7 課）畫出來：顏色越深，10 公里內的其他縣市越多。

In [ ]:
count = dict(zip(df["縣市"], df["10公里內的其他縣市數"]))
gj = simple.to_geojson()
for f in gj["features"]:
    f["properties"]["near"] = count[f["properties"]["COUNTYNAME"]]
result = s.add_geojson(gj, name="10公里內的其他縣市數")
result.style("graduated", field="near", method="manual", breaks=[0, 2, 3, 4, 5, 6])
s.run("labels.field", layer=result.name, field="COUNTYNAME")

for layer in (county, buf, simple):
    layer.visible = False
s.zoom_to((119.3, 21.85, 122.1, 25.35))
s

## 8. 存檔與下載

- `taiwan-counties.rgis`：RustGIS 專案檔，可以用網頁版的 **開啟專案** 打開，繼續在網頁版操作。
- `10km-neighbors.csv`：分析結果的表格，可以用 Excel 打開。

在 Colab 執行時，兩個檔案會自動下載到電腦。

In [ ]:
s.save("taiwan-counties.rgis")
df.to_csv("10km-neighbors.csv", index=False, encoding="utf-8-sig")

try:
    from google.colab import files
    files.download("taiwan-counties.rgis")
    files.download("10km-neighbors.csv")
except ImportError:
    print("不是在 Colab 執行，檔案存在目前的資料夾。")

## 資料來源

內政部國土測繪中心「直轄市、縣市界線(TWD97經緯度)」開放資料（[政府資料開放平臺 7442](https://data.gov.tw/dataset/7442)），依[政府資料開放授權條款－第 1 版](https://data.gov.tw/license)提供。